# 集群平台、网络存储与可靠性补充线 · 第 4/8 课：容器、驱动、Device Plugin 与升级兼容性

> 状态：**参考答案版**  
> 逐课通过制：未完成代码、问答与边界解释前，不进入下一课。

## 本课目标与完成标准

本课产出：实现组件版本区间的兼容性检查，并解释 GPU 设备从节点到容器的分配链。

通过要求：唯一代码填空题通过给定检查；Q1～Q3 都给出因果链；能指出一个正确性边界和一个性能/运营取舍；总分至少 8/10。

## 与既有六条主线的边界

本课不教 Kubernetes 命令清单；重点是 host driver、container runtime、device plugin、CUDA user-space 和作业镜像之间的责任边界。

前置：Linux/网络基础、runtime 补充线、train 分布式章节。本课只补 AI Infra 缺口，不重复已经完成的 kernel 或并行算法推导。

## 核心心智模型

### 它是什么、解决什么问题

Host driver 控制 GPU；容器通常携带用户态 CUDA/库，runtime 注入设备节点与兼容库；device plugin 向 kubelet 广告可分配资源并在 Allocate 阶段提供设备信息。

### 数据与控制如何流动

节点 driver 枚举并管理 GPU，device plugin 向 kubelet 注册健康资源；scheduler 选择节点后，kubelet 调 Allocate，container runtime/CDI 把设备节点和所需库映射进容器，应用再加载用户态 CUDA/NCCL。

### 正确性条件与常见误区

镜像内 `nvidia-smi` 成功不代表 NCCL/RDMA/应用 ABI 全兼容。升级需明确 driver 最低/最高兼容、kernel module、fabric manager、插件与节点排空/回滚。

### 性能、成本与工程取舍

固定镜像提高复现但会积累安全/驱动债务；滚动升级降低爆炸半径，却可能造成集群版本混杂和调度碎片。

## 具体演示

应用要求 driver major 550～570，节点 A=555 可运行，节点 B=545 应被调度过滤，而不是启动后才报符号/设备错误。

请先独立复述“输入 → 状态变化 → 输出/指标”，再开始填空。

## 实践任务：唯一代码填空题

补齐多组件版本约束校验；版本用整数 tuple 比较。

只能修改 `TODO`/`______` 位置，不得删除断言或放宽通过条件。

In [ ]:
def compatible(version, minimum, maximum=None):
    if not version or not minimum:
        raise ValueError("empty version")
    upper_ok = True if maximum is None else version <= maximum
    # TODO：同时满足下界与可选上界。
    return ______

assert compatible((555, 42), (550, 0), (570, 0))
assert not compatible((545, 23), (550, 0), (570, 0))
assert compatible((600, 0), (550, 0), None)


### 检查方法

运行本单元格，所有断言必须通过；再补一个边界输入并解释预期。

### Q1

为什么 CUDA toolkit 在容器内，driver 通常仍必须在 host？

**你的答案：**


### Q2

Device Plugin advertised 8 GPUs，但 Pod 内看不到设备，排查链路是什么？

**你的答案：**


### Q3

GPU 驱动滚动升级为什么需要节点排空与回滚门槛？

**你的答案：**


## 评分规则

- 代码 4 分：正常输入 2 分、边界输入 1 分、解释 1 分；
- Q1～Q3 各 2 分；
- 一票否决：混淆测量与推测、忽略租户/请求隔离、用平均值掩盖尾延迟、删除失败路径。

## 参考答案（仅 answer 分支）

先独立完成。核对后请改变一个规模或故障条件重新推演。

In [ ]:
def compatible(version, minimum, maximum=None):
    if not version or not minimum:
        raise ValueError("empty version")
    upper_ok = True if maximum is None else version <= maximum
    return version >= minimum and upper_ok

assert compatible((555, 42), (550, 0), (570, 0))
assert not compatible((545, 23), (550, 0), (570, 0))
assert compatible((600, 0), (550, 0), None)


### Q1 参考答案

GPU kernel module 和硬件控制属于 host kernel/driver；容器共享 host kernel，只携带用户态 runtime/库。runtime 通过设备节点和库注入连接两者。

### Q2 参考答案

查节点 driver/设备健康、plugin 注册与 Allocate 响应、kubelet resource 分配、container runtime hook/CDI、设备节点/cgroup 权限和 Pod 请求。不能只重启应用容器。

### Q3 参考答案

卸载/重载 kernel module 会影响所有 GPU 工作负载；混部作业可能中断。排空控制影响范围，canary 验证通信/性能，门槛防止新版本造成全池故障。

## 参考资料

- [Kubernetes device plugins](https://kubernetes.io/docs/concepts/extend-kubernetes/compute-storage-net/device-plugins/)
- [NVIDIA GPU Operator](https://docs.nvidia.com/datacenter/cloud-native/gpu-operator/latest/)

API 与平台能力会演进；部署前应按目标版本重新核对。